# 09 — TTS Generation with Kokoro

**Marker:** `NOTEBOOK_09_TTS_GENERATION_KOKORO_V1`

This notebook turns the checked narration selected by Notebook 08 into:

- one WAV file for every script segment
- one combined `narration.wav`
- a plain-text narration transcript
- a timing and traceability manifest

The generated audio stays aligned with the checked script. TTS-only
pronunciation changes are recorded separately in the manifest and do not
modify the source JSON.


## Load the project

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.tts import (
    KokoroSynthesizer,
    find_metadata_file,
    generate_tts_assets,
    load_metadata,
    load_script,
    resolve_checked_script_path,
    summarize_tts_result,
    validate_script_metadata_pair,
)

print("NOTEBOOK_09_TTS_GENERATION_KOKORO_V1")
print(f"Project root: {PROJECT_ROOT}")

## Dependency check

Install the bundled TTS requirements before continuing:

```powershell
pip install -r requirements_tts.txt
```

On Windows, install the 64-bit `espeak-ng` MSI as described in
`INSTALL_AND_RENAME.txt`, then restart the kernel.


In [ ]:
try:
    import kokoro
    import numpy as np
    import soundfile as sf
except ImportError as error:
    raise ImportError(
        "Missing TTS dependency. Install requirements_tts.txt and restart "
        "the Jupyter kernel."
    ) from error

print("Kokoro, NumPy, and SoundFile imported successfully.")

## Configuration

In [ ]:
METADATA_DIRECTORY = PROJECT_ROOT / "data" / "metadata"
CHECKED_SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "checked_scripts"
AUDIO_OUTPUT_DIRECTORY = PROJECT_ROOT / "data" / "audio"

# Leave as None to use the newest metadata JSON.
METADATA_FILENAME = None

# Kokoro language code: "a" is American English; "b" is British English.
LANGUAGE_CODE = "a"

# A configurable starting voice. Try af_heart, af_bella, af_sarah,
# am_adam, or am_michael if you want a different sound.
VOICE = "am_michael"

# Values above 1.0 speak faster; values below 1.0 speak slower.
SPEED = 1.0

SEGMENT_PAUSE_MS = 240
CHUNK_PAUSE_MS = 80
TARGET_PEAK_DBFS = -1.0
TRIM_EDGE_SILENCE = True
NORMALIZE_CONTEXTUAL_YEARS = True

# Add exact text replacements only when a name, acronym, or number is
# pronounced incorrectly. These affect TTS text, not the checked script.
PRONUNCIATION_REPLACEMENTS = {
    # "AROSICS": "A row six",
}

print(f"Metadata directory: {METADATA_DIRECTORY}")
print(f"Checked scripts: {CHECKED_SCRIPTS_DIRECTORY}")
print(f"Audio output: {AUDIO_OUTPUT_DIRECTORY}")
print(f"Voice: {VOICE}")
print(f"Speed: {SPEED}")

## Load the newest metadata and its checked script

In [ ]:
metadata_path = find_metadata_file(
    metadata_directory=METADATA_DIRECTORY,
    filename=METADATA_FILENAME,
)
metadata = load_metadata(metadata_path)

checked_script_path = resolve_checked_script_path(
    metadata=metadata,
    checked_scripts_directory=CHECKED_SCRIPTS_DIRECTORY,
)
checked_script = load_script(checked_script_path)

validate_script_metadata_pair(
    script=checked_script,
    metadata=metadata,
)

print(f"Metadata: {metadata_path}")
print(f"Checked script: {checked_script_path}")
print(f"Topic: {checked_script.topic.title}")
print(f"Words: {checked_script.word_count}")
print(f"Estimated seconds: {checked_script.estimated_total_seconds}")
print(f"Fact-check verdict: {metadata.fact_check_verdict}")
print(f"Manual review recommended: {metadata.requires_manual_review}")

## Preview the source narration

In [ ]:
print(f"HOOK\n{checked_script.hook.narration}\n")

for index, section in enumerate(checked_script.sections, start=1):
    print(
        f"{index}. {section.segment_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(section.narration)
    print()

print(
    f"CLOSING ({checked_script.closing.estimated_seconds}s)\n"
    f"{checked_script.closing.narration}"
)

## Initialize Kokoro

The first initialization may take longer because model files can be downloaded
and cached locally.


In [ ]:
synthesizer = KokoroSynthesizer(
    language_code=LANGUAGE_CODE,
    voice=VOICE,
    speed=SPEED,
    chunk_pause_ms=CHUNK_PAUSE_MS,
)

print(
    f"Loaded {synthesizer.model_name} "
    f"with voice {synthesizer.voice}."
)

## Generate segment and master audio

In [ ]:
tts_result = generate_tts_assets(
    script=checked_script,
    metadata=metadata,
    metadata_filename=metadata_path.name,
    output_root=AUDIO_OUTPUT_DIRECTORY,
    synthesizer=synthesizer,
    segment_pause_ms=SEGMENT_PAUSE_MS,
    pronunciation_replacements=PRONUNCIATION_REPLACEMENTS,
    normalize_contextual_years=NORMALIZE_CONTEXTUAL_YEARS,
    trim_edge_silence=TRIM_EDGE_SILENCE,
    target_peak_dbfs=TARGET_PEAK_DBFS,
)

for name, value in summarize_tts_result(tts_result).items():
    print(f"{name}: {value}")

## Timing review

In [ ]:
for segment in tts_result.manifest.segments:
    difference = (
        segment.actual_seconds - segment.estimated_seconds
    )

    print(
        f"{segment.index:02d} {segment.segment_type}: "
        f"estimated={segment.estimated_seconds:.1f}s, "
        f"actual={segment.actual_seconds:.2f}s, "
        f"difference={difference:+.2f}s"
    )

ratio = tts_result.manifest.duration_ratio

if ratio < 0.75:
    print(
        "\nWARNING: The TTS is much faster than the script estimate. "
        "Consider lowering SPEED."
    )
elif ratio > 1.25:
    print(
        "\nWARNING: The TTS is much slower than the script estimate. "
        "Consider raising SPEED or shortening the script."
    )
else:
    print("\nOverall TTS duration is reasonably close to the estimate.")

## Listen to the complete narration

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        filename=str(tts_result.master_audio_path),
        autoplay=False,
    )
)

## Listen to individual segments

In [ ]:
for segment, path in zip(
    tts_result.manifest.segments,
    tts_result.segment_paths,
    strict=True,
):
    print(
        f"{segment.index:02d} — {segment.segment_type} "
        f"({segment.actual_seconds:.2f}s)"
    )
    display(Audio(filename=str(path), autoplay=False))

## Inspect TTS-only text and output files

In [ ]:
for segment in tts_result.manifest.segments:
    print(f"{segment.index:02d} {segment.segment_type}")
    print(f"Source: {segment.source_text}")
    print(f"TTS:    {segment.tts_text}")
    print(f"File:   {segment.audio_filename}")
    print()

print(f"Master audio: {tts_result.master_audio_path}")
print(f"Transcript: {tts_result.transcript_path}")
print(f"Manifest: {tts_result.manifest_path}")

## Preview the complete manifest

In [ ]:
print(tts_result.manifest.model_dump_json(indent=2))